# 01 - One-time AWS bootstrap

Idempotent setup for the pruning-metrics workflow. Run this notebook
once per AWS account / region. It creates (or refreshes):

* an S3 bucket for code tarballs, pruning calibration artifacts,
  free-form eval results, and teacher-forced records;
* an IAM role (`pruning-metrics-ec2`) trusted by `ec2.amazonaws.com`
  with scoped S3 + SSM Session Manager + CloudWatch agent
  permissions;
* the matching IAM instance profile (same name).

Re-running is a no-op for existing resources. Notebook 2 onwards
assume these resources exist.


In [1]:
import os
import sys
from pathlib import Path

# Allow the notebook to be run from anywhere by pinning to the repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Optional: load .env so AWS_PROFILE etc. surface in the kernel.
try:
    from dotenv import load_dotenv

    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))
print("AWS_REGION  =", os.environ.get("AWS_REGION"))


REPO_ROOT = /var/home/three-kingdoms/work/pruning-metrics
AWS_PROFILE = rengz
AWS_REGION  = us-east-1


## Configuration

All knobs live in this cell. The defaults match what the rest of
the project expects, so most users only edit `RESULTS_BUCKET`
(must be globally unique, follow S3 naming rules).


In [2]:
AWS_PROFILE = os.environ.get("AWS_PROFILE", "rengz")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "414266451290")
RESULTS_BUCKET = os.environ.get(
    "RESULTS_BUCKET", f"pruning-metrics-results-{ACCOUNT_ID}"
)
EC2_INSTANCE_ROLE_NAME = os.environ.get(
    "EC2_INSTANCE_ROLE_NAME", "pruning-metrics-ec2"
)
print({
    "AWS_PROFILE": AWS_PROFILE,
    "AWS_REGION": AWS_REGION,
    "ACCOUNT_ID": ACCOUNT_ID,
    "RESULTS_BUCKET": RESULTS_BUCKET,
    "EC2_INSTANCE_ROLE_NAME": EC2_INSTANCE_ROLE_NAME,
})


{'AWS_PROFILE': 'rengz', 'AWS_REGION': 'us-east-1', 'ACCOUNT_ID': '414266451290', 'RESULTS_BUCKET': 'pruning-metrics-results-414266451290', 'EC2_INSTANCE_ROLE_NAME': 'pruning-metrics-ec2'}


## Verify AWS credentials

STS `GetCallerIdentity` confirms the kernel can reach AWS APIs and
displays the assumed identity (the SSO role for `rengz`).


In [3]:
import json
import boto3

session = boto3.session.Session(profile_name=AWS_PROFILE)
sts = session.client("sts", region_name=AWS_REGION)
identity = sts.get_caller_identity()
print(json.dumps(identity, indent=2, default=str))
assert identity["Account"] == ACCOUNT_ID, (
    f"Expected account {ACCOUNT_ID}, got {identity['Account']}"
)


{
  "UserId": "AROAWA5BUMVNPTLBG6UFS:rengz@mit.edu",
  "Account": "414266451290",
  "Arn": "arn:aws:sts::414266451290:assumed-role/IdP-admin-role/rengz@mit.edu",
  "ResponseMetadata": {
    "RequestId": "0837b2c3-b798-463c-80cb-7c6559cf1b3a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "0837b2c3-b798-463c-80cb-7c6559cf1b3a",
      "x-amz-sts-extended-request-id": "MTp1cy1lYXN0LTE6UzoxNzc3ODYwMjIzMTM5OlI6RmFyaVpoWE0=",
      "content-type": "text/xml",
      "content-length": "447",
      "date": "Mon, 04 May 2026 02:03:43 GMT"
    },
    "RetryAttempts": 0
  }
}


## Run the bootstrap script

`infra/aws/setup/bootstrap_ec2_resources.py` is idempotent. It
checks for the bucket / role / profile and creates anything
missing. Output below is a single JSON summary.


In [4]:
import subprocess

cmd = [
    sys.executable,
    str(REPO_ROOT / "infra" / "aws" / "setup" / "bootstrap_ec2_resources.py"),
    "--bucket", RESULTS_BUCKET,
    "--region", AWS_REGION,
    "--role-name", EC2_INSTANCE_ROLE_NAME,
]
env = dict(os.environ)
env["AWS_PROFILE"] = AWS_PROFILE
completed = subprocess.run(cmd, env=env, check=True, capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print("--- stderr ---")
    print(completed.stderr)


S3 bucket already exists: pruning-metrics-results-414266451290
IAM role already exists: arn:aws:iam::414266451290:role/pruning-metrics-ec2
Attached inline ResultsBucketAccess policy to pruning-metrics-ec2.
Attached managed policy arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore to pruning-metrics-ec2.
Attached managed policy arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy to pruning-metrics-ec2.
Instance profile already exists: pruning-metrics-ec2
Role pruning-metrics-ec2 already attached to profile pruning-metrics-ec2.
{
  "bucket": "pruning-metrics-results-414266451290",
  "region": "us-east-1",
  "role_name": "pruning-metrics-ec2",
  "role_arn": "arn:aws:iam::414266451290:role/pruning-metrics-ec2",
  "instance_profile_name": "pruning-metrics-ec2"
}



## Verify the resources

Confirms the bucket exists, the role exists and is trusted by EC2,
and the instance profile carries the role.


In [5]:
s3 = session.client("s3", region_name=AWS_REGION)
iam = session.client("iam")

head = s3.head_bucket(Bucket=RESULTS_BUCKET)
print("OK S3 bucket:", RESULTS_BUCKET)

role = iam.get_role(RoleName=EC2_INSTANCE_ROLE_NAME)["Role"]
print("OK IAM role:", role["Arn"])

profile = iam.get_instance_profile(InstanceProfileName=EC2_INSTANCE_ROLE_NAME)["InstanceProfile"]
attached_roles = [r["RoleName"] for r in profile.get("Roles", [])]
assert EC2_INSTANCE_ROLE_NAME in attached_roles, attached_roles
print("OK Instance profile:", profile["Arn"])


OK S3 bucket: pruning-metrics-results-414266451290


OK IAM role: arn:aws:iam::414266451290:role/pruning-metrics-ec2
OK Instance profile: arn:aws:iam::414266451290:instance-profile/pruning-metrics-ec2


## Summary

Bucket and instance profile are ready. Use the values above as
inputs to notebooks 2/3/4. The instance profile name is also
stored in `.env` as `EC2_INSTANCE_PROFILE_NAME` so the launch
scripts pick it up automatically.


In [6]:
print("Setup complete. Use these values in subsequent notebooks:")
print(json.dumps({
    "AWS_PROFILE": AWS_PROFILE,
    "AWS_REGION": AWS_REGION,
    "RESULTS_BUCKET": RESULTS_BUCKET,
    "EC2_INSTANCE_PROFILE_NAME": EC2_INSTANCE_ROLE_NAME,
}, indent=2))


Setup complete. Use these values in subsequent notebooks:
{
  "AWS_PROFILE": "rengz",
  "AWS_REGION": "us-east-1",
  "RESULTS_BUCKET": "pruning-metrics-results-414266451290",
  "EC2_INSTANCE_PROFILE_NAME": "pruning-metrics-ec2"
}
